# Visualize PHY topology and ILP paths

Chỉnh các biến trong cell **Configuration**, sau đó chạy lần lượt các cell. Notebook hỗ trợ vẽ toàn mạng, một candidate path, toàn bộ path trong nghiệm và placement của một NSC.

In [ ]:
from pathlib import Path
import sys
from IPython.display import Image, display

def find_repo_root():
    candidates = (Path.cwd(), *Path.cwd().parents)
    for candidate in candidates:
        if (candidate / 'FlexRanSplitProblem').is_dir():
            return candidate
    raise FileNotFoundError('Cannot find the Flex-split repository root')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from FlexRanSplitProblem.visualize_phy import (
    draw_phy_and_paths,
    edge_sequence,
    find_path,
    load_pickle,
    load_slice_solution,
    load_solution,
)

print('Repository root:', REPO_ROOT)

## Configuration

In [ ]:
NETWORK_NAME = 'network_16'
CORE_ID = 0
PATH_ID = 5
SLICE_ID = 1
SHOW_RESOURCES = True

PHY_PATH = REPO_ROOT / 'data' / NETWORK_NAME / 'phy.pkl'
PATHS_PATH = REPO_ROOT / 'data' / NETWORK_NAME / 'paths.pkl'
RESULTS_DIR = REPO_ROOT / 'results' / NETWORK_NAME
OUTPUT_DIR = RESULTS_DIR / 'visualizations'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

solution_files = sorted(RESULTS_DIR.glob('solution_*.json'))
SOLUTION_PATH = solution_files[-1] if solution_files else None

print('PHY      :', PHY_PATH)
print('Paths    :', PATHS_PATH)
print('Solution :', SOLUTION_PATH or 'No saved solution found')

## Load data

In [ ]:
physical_network = load_pickle(PHY_PATH)
paths = load_pickle(PATHS_PATH)

ru_nodes = sorted({path.nodes[-1] for path in paths})
print(f'PHY: {physical_network.number_of_nodes()} nodes, {physical_network.number_of_edges()} directed links')
print(f'Candidate paths: {len(paths)}')
print(f'RU endpoints: {ru_nodes}')

## 1. Draw the complete PHY network

In [ ]:
phy_image = draw_phy_and_paths(
    physical_network=physical_network,
    paths=paths,
    core_id=CORE_ID,
    node_resources=SHOW_RESOURCES,
    edge_labels=SHOW_RESOURCES,
    output_path=OUTPUT_DIR / 'phy.png',
    title=f'{NETWORK_NAME}: physical RAN topology',
)
display(Image(filename=str(phy_image)))

## 2. Highlight one candidate path

In [ ]:
selected_path = find_path(paths, PATH_ID)
print('Path ID :', selected_path.id)
print('Nodes   :', selected_path.nodes)
print('Links   :', edge_sequence(selected_path.nodes))
print('Counts  :', len(selected_path.nodes), 'nodes,', len(selected_path.nodes) - 1, 'links')

path_image = draw_phy_and_paths(
    physical_network=physical_network,
    paths=paths,
    path_ids=[PATH_ID],
    core_id=CORE_ID,
    node_resources=SHOW_RESOURCES,
    edge_labels=SHOW_RESOURCES,
    output_path=OUTPUT_DIR / f'path_{PATH_ID}.png',
    title=f'{NETWORK_NAME}: candidate path {PATH_ID}',
)
display(Image(filename=str(path_image)))

## 3. Draw every selected path from a saved ILP solution

In [ ]:
if SOLUTION_PATH is None:
    print('Run run_ilp first to create a solution JSON in', RESULTS_DIR)
else:
    solution = load_solution(SOLUTION_PATH)
    selected_path_ids = [int(item['path']) for item in solution]
    print('Selected paths:', selected_path_ids)
    solution_image = draw_phy_and_paths(
        physical_network=physical_network,
        paths=paths,
        path_ids=selected_path_ids,
        core_id=CORE_ID,
        node_resources=SHOW_RESOURCES,
        edge_labels=SHOW_RESOURCES,
        output_path=OUTPUT_DIR / 'solution_paths.png',
        title=f'{NETWORK_NAME}: selected paths for {len(solution)} NSCs',
    )
    display(Image(filename=str(solution_image)))

## 4. Draw one NSC with its VNF placement

In [ ]:
if SOLUTION_PATH is None:
    print('Run run_ilp first to create a solution JSON in', RESULTS_DIR)
else:
    selected = load_slice_solution(SOLUTION_PATH, SLICE_ID)
    print('NSC       :', selected['slice'])
    print('DRC       :', selected['drc'])
    print('Path      :', selected['path'])
    print('Path nodes:', selected['path_nodes'])
    print('Placement :', selected['vnf_placement'])
    print('Flows     :', selected['virtual_link_mapping'])
    nsc_image = draw_phy_and_paths(
        physical_network=physical_network,
        paths=paths,
        path_ids=[int(selected['path'])],
        placement=selected['vnf_placement'],
        core_id=CORE_ID,
        node_resources=SHOW_RESOURCES,
        edge_labels=SHOW_RESOURCES,
        output_path=OUTPUT_DIR / f'nsc_{SLICE_ID}.png',
        title=(
            f"NSC {SLICE_ID}: DRC {selected['drc']}, "
            f"path {selected['path']}, latency {selected['latency']:.3f}"
        ),
    )
    display(Image(filename=str(nsc_image)))